# Cómputo Evolutivo — Clases Base
**Referencia:** Kuri, Á. (2002). *Algoritmos genéticos*. Instituto Politécnico Nacional.

Este notebook modela los conceptos fundamentales derivados de los temas:
- **1.1 La Naturaleza como Optimizadora** — La evolución biológica actúa como un proceso de búsqueda que maximiza la aptitud de los organismos ante su entorno. Un algoritmo genético imita ese proceso para resolver problemas de optimización.
- **1.2 Un poco de Biología** — Los individuos de una población poseen un genotipo (cromosoma) codificado en genes; la selección natural favorece a los más aptos (mayor *fitness*); la reproducción con cruzamiento y mutación genera diversidad.

## Clase `Individuo`

Representa un organismo dentro de la población.  
Su **cromosoma** es la codificación del genotipo (lista de genes/bits).  
Su **aptitud** (*fitness*) mide qué tan bien ese genotipo resuelve el problema (análogo a la supervivencia en la naturaleza).

In [13]:
import random

class Individuo:
    """
    Modela un organismo dentro de una población evolutiva.

    Atributos
    ---------
    cromosoma : list[int]
        Secuencia de genes (bits 0/1) que representan el genotipo.
    longitud  : int
        Número de genes en el cromosoma.
    aptitud   : float
        Valor de fitness asignado por la función objetivo.
        Emula la presión selectiva del entorno natural.
    """

    def __init__(self, longitud: int, cromosoma: list = None):
        self.longitud  = longitud
        # Si no se proporciona cromosoma, se genera uno aleatorio (variación inicial)
        self.cromosoma = cromosoma if cromosoma else self._generar_cromosoma()
        self.aptitud   = 0.0

    # ------------------------------------------------------------------
    # Métodos privados
    # ------------------------------------------------------------------

    def _generar_cromosoma(self) -> list:
        """Genera un genotipo binario aleatorio (diversidad inicial)."""
        return [random.randint(0, 1) for _ in range(self.longitud)]

    # ------------------------------------------------------------------
    # Métodos públicos
    # ------------------------------------------------------------------

    def evaluar(self, funcion_objetivo) -> float:
        """
        Calcula la aptitud del individuo aplicando la función objetivo.
        Simula la 'prueba' que el entorno le impone al organismo.
        """
        self.aptitud = funcion_objetivo(self.cromosoma)
        return self.aptitud

    def mutar(self, tasa_mutacion: float = 0.01) -> None:
        """
        Aplica mutación gen a gen con la tasa dada.
        Introduce pequeñas variaciones aleatorias (análogo a errores de copia en el ADN).
        """
        for i in range(self.longitud):
            if random.random() < tasa_mutacion:
                self.cromosoma[i] ^= 1  # flip del bit
        # La aptitud ya no es válida tras la mutación
        self.aptitud = 0.0

    def fenotipo(self) -> int:
        """
        Convierte el genotipo binario a su valor decimal (expresión del fenotipo).
        En biología, el fenotipo es la manifestación observable del genotipo.
        """
        return int("".join(str(g) for g in self.cromosoma), 2)

    def __repr__(self) -> str:
        genes = "".join(str(g) for g in self.cromosoma)
        return f"Individuo(genotipo={genes}, fenotipo={self.fenotipo()}, aptitud={self.aptitud:.4f})"


## Clase `Poblacion`

Agrupa a un conjunto de individuos que compiten y se reproducen entre sí.  
Modela los conceptos de:
- **Generación** — ciclo de evaluación → selección → reproducción.
- **Selección natural** — se favorece a los individuos con mayor aptitud.
- **Cruzamiento** — combinación de material genético de dos progenitores.

In [14]:
class Poblacion:
    """
    Modela un conjunto de individuos sometidos a la presión evolutiva.

    Atributos
    ---------
    tamanio          : int
        Número de individuos en la población.
    longitud_cromosoma : int
        Longitud del cromosoma de cada individuo.
    individuos       : list[Individuo]
        Colección actual de organismos.
    generacion       : int
        Contador del ciclo evolutivo actual.
    mejor            : Individuo | None
        El individuo con mayor aptitud encontrado hasta ahora.
    """

    def __init__(self, tamanio: int, longitud_cromosoma: int):
        self.tamanio             = tamanio
        self.longitud_cromosoma  = longitud_cromosoma
        self.generacion          = 0
        self.mejor               = None
        # Genera la población inicial con diversidad aleatoria
        self.individuos = [Individuo(longitud_cromosoma) for _ in range(tamanio)]

    # ------------------------------------------------------------------
    # Métodos privados
    # ------------------------------------------------------------------

    def _seleccion_ruleta(self) -> Individuo:
        """
        Selección proporcional a la aptitud (ruleta de fitness).
        Los individuos más aptos tienen mayor probabilidad de reproducirse,
        imitando la selección natural de Darwin.
        """
        aptitud_total = sum(ind.aptitud for ind in self.individuos)
        if aptitud_total == 0:
            return random.choice(self.individuos)
        punto = random.uniform(0, aptitud_total)
        acumulado = 0.0
        for ind in self.individuos:
            acumulado += ind.aptitud
            if acumulado >= punto:
                return ind
        return self.individuos[-1]

    def _cruzar(self, padre: Individuo, madre: Individuo) -> tuple:
        """
        Cruzamiento de un punto: combina los cromosomas de dos progenitores
        para producir dos descendientes (análogo a la meiosis biológica).
        """
        punto = random.randint(1, self.longitud_cromosoma - 1)
        crom_hijo1 = padre.cromosoma[:punto] + madre.cromosoma[punto:]
        crom_hijo2 = madre.cromosoma[:punto] + padre.cromosoma[punto:]
        return Individuo(self.longitud_cromosoma, crom_hijo1), \
               Individuo(self.longitud_cromosoma, crom_hijo2)

    # ------------------------------------------------------------------
    # Métodos públicos
    # ------------------------------------------------------------------

    def evaluar_poblacion(self, funcion_objetivo) -> None:
        """
        Evalúa la aptitud de todos los individuos.
        Actualiza el registro del mejor individuo de la historia.
        """
        for ind in self.individuos:
            ind.evaluar(funcion_objetivo)
        candidato = max(self.individuos, key=lambda i: i.aptitud)
        if self.mejor is None or candidato.aptitud > self.mejor.aptitud:
            # Guardar copia del mejor para no perderlo en la siguiente generación
            self.mejor = Individuo(self.longitud_cromosoma, list(candidato.cromosoma))
            self.mejor.aptitud = candidato.aptitud

    def evolucionar(self, funcion_objetivo, tasa_mutacion: float = 0.01) -> None:
        """
        Ejecuta un ciclo generacional completo:
          1. Evaluación de aptitud
          2. Selección natural (ruleta)
          3. Cruzamiento (reproducción)
          4. Mutación (variación)
        """
        self.evaluar_poblacion(funcion_objetivo)

        nueva_generacion = []
        while len(nueva_generacion) < self.tamanio:
            padre = self._seleccion_ruleta()
            madre = self._seleccion_ruleta()
            hijo1, hijo2 = self._cruzar(padre, madre)
            hijo1.mutar(tasa_mutacion)
            hijo2.mutar(tasa_mutacion)
            nueva_generacion.extend([hijo1, hijo2])

        self.individuos = nueva_generacion[:self.tamanio]
        self.evaluar_poblacion(funcion_objetivo)
        self.generacion += 1

    def estadisticas(self) -> dict:
        """Resumen estadístico de la generación actual."""
        aptitudes = [ind.aptitud for ind in self.individuos]
        return {
            "generacion"  : self.generacion,
            "aptitud_max" : max(aptitudes),
            "aptitud_min" : min(aptitudes),
            "aptitud_prom": sum(aptitudes) / len(aptitudes),
        }

    def __repr__(self) -> str:
        return (f"Poblacion(tamanio={self.tamanio}, "
                f"generacion={self.generacion}, "
                f"mejor_aptitud={self.mejor.aptitud if self.mejor else 'N/A'})")


## Demostración

**Problema:** maximizar la cantidad de `1`s en el cromosoma (*OneMax*).  
Es el problema más sencillo de optimización binaria y sirve para verificar que el AG converge correctamente.

In [15]:
# Función objetivo: contar cuántos genes son 1 (maximizar)
def onemax(cromosoma: list) -> float:
    return float(sum(cromosoma))

# Parámetros
TAMANIO_POBLACION  = 90
LONGITUD_CROMOSOMA = 10
GENERACIONES       = 60
TASA_MUTACION      = 0.02

# Crear población inicial
poblacion = Poblacion(TAMANIO_POBLACION, LONGITUD_CROMOSOMA)
print(f"Estado inicial: {poblacion}\n")

# Ciclo evolutivo
for _ in range(GENERACIONES):
    poblacion.evolucionar(onemax, TASA_MUTACION)
    stats = poblacion.estadisticas()
    print(f"Gen {stats['generacion']:>2} | "
          f"Max: {stats['aptitud_max']:.0f}  "
          f"Prom: {stats['aptitud_prom']:.2f}  "
          f"Min: {stats['aptitud_min']:.0f}")

print(f"\nMejor individuo encontrado: {poblacion.mejor}")


Estado inicial: Poblacion(tamanio=90, generacion=0, mejor_aptitud=N/A)

Gen  1 | Max: 9  Prom: 5.98  Min: 3
Gen  2 | Max: 10  Prom: 6.19  Min: 2
Gen  3 | Max: 10  Prom: 6.58  Min: 3
Gen  4 | Max: 10  Prom: 6.73  Min: 3
Gen  5 | Max: 10  Prom: 7.24  Min: 4
Gen  6 | Max: 10  Prom: 7.07  Min: 4
Gen  7 | Max: 10  Prom: 7.32  Min: 4
Gen  8 | Max: 10  Prom: 7.47  Min: 4
Gen  9 | Max: 10  Prom: 7.73  Min: 4
Gen 10 | Max: 10  Prom: 7.59  Min: 4
Gen 11 | Max: 10  Prom: 7.74  Min: 5
Gen 12 | Max: 10  Prom: 7.90  Min: 5
Gen 13 | Max: 10  Prom: 7.88  Min: 5
Gen 14 | Max: 10  Prom: 7.89  Min: 5
Gen 15 | Max: 10  Prom: 7.93  Min: 5
Gen 16 | Max: 10  Prom: 7.91  Min: 5
Gen 17 | Max: 10  Prom: 7.78  Min: 4
Gen 18 | Max: 10  Prom: 7.92  Min: 5
Gen 19 | Max: 10  Prom: 7.89  Min: 5
Gen 20 | Max: 10  Prom: 7.94  Min: 5
Gen 21 | Max: 10  Prom: 8.11  Min: 4
Gen 22 | Max: 10  Prom: 7.99  Min: 5
Gen 23 | Max: 10  Prom: 8.03  Min: 4
Gen 24 | Max: 10  Prom: 8.06  Min: 5
Gen 25 | Max: 10  Prom: 8.39  Min: 5
Gen 